#This is the word count spark logic, counting words from a book

In [ ]:
from pyspark import SparkContext, SparkConf
from prettytable import PrettyTable
import re

conf = SparkConf().setMaster("local[*]").setAppName("WordCount")
sc = SparkContext.getOrCreate(conf)
# sc = SparkContext(conf = conf)

WORD_RE = re.compile(r"\b\w+(?: '\w+)*\b", re.UNICODE) # This word may or may not have punctuation at the end, but it must be a word character (letter, digit, or underscore) at the beginning and end. The optional apostrophe allows for contractions like "don't" or possessives like "John's".

def normalizeWords(line):
    return WORD_RE.findall(line.lower())

books = sc.textFile('./pride_and_prejudice.txt')
words = books.flatMap(normalizeWords)
# wordCounts = words.countByValue()
wordCounts = words.map(lambda x: (x, 1)).reduceByKey(lambda x, y: x + y)
wordCountsSorted = wordCounts.sortBy(lambda x: x[1], ascending=False)


table = PrettyTable()
table.field_names = ["Word", "Count"]

result = wordCountsSorted.collect()

for (word, count) in result:
    table.add_row([word, count])

print(table)
sc.stop()

+------+-------+
| Word | Count |
+------+-------+
+------+-------+


This is customer id and the total of the orders on the right in descending order

In [ ]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

conf = SparkConf().setMaster("local[*]").setAppName("CustomerTotalsSorted")
sc = SparkContext.getOrCreate(conf)

lines = sc.textFile("customer-orders.csv")

rdd = lines.map(lambda line: (
    line.split(',')[0],
    float(line.split(',')[2])
))

totals_by_customer = rdd.reduceByKey(lambda x, y: x + y)
sorted_totals = totals_by_customer.sortBy(lambda x: x[1], ascending=False)
results = sorted_totals.collect()

table = PrettyTable()
table.field_names = ["Customer ID", "Total Spent"]

for customer_id, total in results:
    table.add_row([customer_id, round(total, 2)])

print(table)

# sc.stop()